# 5 — DeBERTa / NLI faithfulness baseline

Thesis **baseline** filter: fine-tune DeBERTa (`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`) as a binary faithfulness classifier (premise = context, hypothesis = answer), compared against zero-shot NLI.

**Protocol**
- Dataset: `data/labeled_merged.csv`
- Leakage-safe base-ID split (`asqa_X` / `asqa_X_hallu` stay together)
- `test_size=0.2`, `random_state=42` → freeze `data/labeled_merged_test.csv`
- Val carved from remaining train for min-FPR threshold @ recall ≥ 0.70
- Train DeBERTa **3 identical runs** (same split/seed); report mean±std
- Compare: No Filter / NLI zero-shot / DeBERTa

Heavy lifting lives in `src/filtering/` and `scripts/run_deberta_nli_baseline.py`. This notebook is the interactive driver.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# ========== 1) SET THIS IF NEEDED ==========
# On Kaggle (clone into /kaggle/working/ragas-evaluation):
REPO_ROOT = Path("/kaggle/working/ragas-evaluation")
# On local machine (notebook inside notebooks/):
if not (REPO_ROOT / "scripts" / "run_deberta_nli_baseline.py").is_file():
    REPO_ROOT = Path("..").resolve()

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT =", REPO_ROOT)
assert (REPO_ROOT / "scripts" / "run_deberta_nli_baseline.py").is_file(), (
    "Script not found. Fix REPO_ROOT. Expected: "
    f"{REPO_ROOT}/scripts/run_deberta_nli_baseline.py"
)

# ========== 2) Project imports ==========
# If error says: partially initialized module 'torch' (circular import)
# -> menu: Restart session / Restart kernel, then run Setup again.

from src.filtering.config_loader import load_yaml, resolve_path
from src.filtering.data_split import load_and_split, to_base_id
from src.filtering.learned_filter import (
    _extract_top1_context,
    _truncation_collision_diagnostic,
    overfit_sanity_check,
)
from transformers import AutoTokenizer

CFG_PATH = REPO_ROOT / "configs" / "experiments" / "filter_training.yaml"
cfg = load_yaml(str(CFG_PATH))
data_cfg = cfg["data"]
print("OK — ready. n_runs =", cfg.get("n_runs", 3))


## 2. Leakage-safe split + freeze test CSV

Same intent as:
```python
train_test_split(df, test_size=0.2, random_state=42)
```
but split on **base IDs** so correct/hallu pairs never cross the train–test boundary.

In [ ]:
train_df, val_df, test_df = load_and_split(
    csv_path=str(resolve_path(data_cfg["labeled_csv"])),
    test_ratio=data_cfg["test_ratio"],
    val_ratio=data_cfg["val_ratio"],
    seed=data_cfg["seed"],
    test_csv_path=str(resolve_path(data_cfg["test_csv"])) if data_cfg.get("test_csv") else None,
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: n={len(split)} pos={(split.label==1).sum()} neg={(split.label==0).sum()} bases={split.id.map(to_base_id).nunique()}")

assert set(test_df.id.map(to_base_id)).isdisjoint(set(train_df.id.map(to_base_id)))
assert set(test_df.id.map(to_base_id)).isdisjoint(set(val_df.id.map(to_base_id)))
print("OK: no base-ID leakage across splits")
print("Frozen test CSV:", resolve_path(data_cfg["test_csv"]))

## 3. Pre-training gates

Run in order: label check → pair spot-check → truncation diagnostic → overfit sanity check.

In [ ]:
# Gate 1 — label mapping
assert set(train_df["label"].unique()) <= {0, 1}
pos = train_df[train_df.label == 1].iloc[0]
neg = train_df[train_df.id.map(to_base_id) == to_base_id(pos.id)]
neg = neg[neg.label == 0].iloc[0]
print("label==1 (correct) id:", pos.id)
print("label==0 (hallu)   id:", neg.id)
print("question:", pos.question[:120])
print("correct answer:", str(pos.answer)[:160])
print("hallu answer:  ", str(neg.answer)[:160])

In [ ]:
# Gate 2 — five paired spot-checks
shown = 0
for base, group in train_df.groupby(train_df.id.map(to_base_id)):
    if not ((group.label == 1).any() and (group.label == 0).any()):
        continue
    c = group[group.label == 1].iloc[0]
    h = group[group.label == 0].iloc[0]
    print("=" * 60)
    print("base:", base)
    print("Q:", c.question[:100])
    print("ctx:", _extract_top1_context(c.context)[:160])
    print("OK :", str(c.answer)[:120])
    print("BAD:", str(h.answer)[:120])
    shown += 1
    if shown >= 5:
        break

In [ ]:
# Gate 3 — truncation collision (should be < 5%)
from src.filtering.config_loader import load_config_section, FILTERING_CONFIG

lf_cfg = load_config_section(FILTERING_CONFIG, "learned_filter")
tok = AutoTokenizer.from_pretrained(lf_cfg["model_name"])
contexts = [_extract_top1_context(c) for c in train_df["context"].tolist()]
rate = _truncation_collision_diagnostic(
    tok, train_df, contexts, max_length=lf_cfg.get("max_length", 512), n=200,
)
print(f"truncation collision rate = {rate:.3f}")
assert rate <= 0.05, "Raise max_length — discriminative answer tokens are being truncated"

In [ ]:
# Gate 4 — HARD GATE: overfit 16 pairs (32 samples)
gate = overfit_sanity_check(train_df, n_pairs=16, epochs=50)
print(gate)
assert gate["train_f1"] >= 0.95, "Overfit gate failed — do not scale to full training"

## 4. Train (3 runs)

1. **Restart kernel**
2. Run **Setup** (section 1)
3. Run gates (section 3) optional but recommended
4. Run the next cell

Results: `results/deberta_nli/summary_classification_report.csv`


In [ ]:
# ========== TRAIN (3 DeBERTa runs + NLI) ==========
# Restart kernel, run Setup cell above, then this cell.

import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/ragas-evaluation")
if not (REPO_ROOT / "scripts" / "run_deberta_nli_baseline.py").is_file():
    REPO_ROOT = Path("..").resolve()

script = REPO_ROOT / "scripts" / "run_deberta_nli_baseline.py"
cfg = REPO_ROOT / "configs" / "experiments" / "filter_training.yaml"

print("Running:", script)
print("Config :", cfg)
subprocess.check_call(
    [
        sys.executable,
        str(script),
        "--config",
        str(cfg),
        "--skip-overfit-gate",
    ],
    cwd=str(REPO_ROOT),
)
print("Done. See results/deberta_nli/summary_classification_report.csv")


## 5. Load summary + comparison table

In [ ]:
summary_path = resolve_path(cfg["results_dir"]) / "summary.json"
report_path = resolve_path(cfg["results_dir"]) / "summary_classification_report.csv"

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("n_train/val/test:", summary["n_train"], summary["n_val"], summary["n_test"])
    print("\n=== comparison_table ===")
    print(json.dumps(summary.get("comparison_table"), indent=2))
    if "deberta_mean_std" in summary:
        print(pd.DataFrame(summary["deberta_mean_std"]).T)
else:
    print("No summary yet at", summary_path)
    print("Run: python scripts/run_deberta_nli_baseline.py")

# Same schema as RAGAS / LLM-judge summaries:
# dataset,num_samples,accepted,acceptance_rate,accuracy,precision,recall,f1,roc_auc
if report_path.exists():
    report_df = pd.read_csv(report_path)
    print("\n=== summary_classification_report (DeBERTa mean over runs) ===")
    display(report_df)
else:
    print("\nNo classification report yet at", report_path)
